In [ ]:
import polars as pl
from pathlib import Path

data_dir = Path().absolute() / ".." / "data"
df = pl.read_parquet(data_dir / "parsed_conversations/cm0i27jdj0000aqpa73ghpcxf.snappy")
# df.to_pandas().to_csv(data_dir / "parsed_conversations/cm0i27jdj0000aqpa73ghpcxf.csv")

In [ ]:
from pathlib import Path
import polars as pl
from dtale import show
import dtale.global_state as global_state

global_state.set_app_settings(dict(max_column_width=300))

data_dir = Path().absolute() / ".." / "data"

In [ ]:
df = pl.read_parquet(data_dir / "conversation_skeletons/cm0i27jdj0000aqpa73ghpcxf.snappy")
d = show(df.to_pandas())
d.open_browser()

In [ ]:
linebreak = "\n"
print(f"""
OBSERVABLES: {linebreak.join(df[0]['observables'].to_list()[0])}\n
INFERRABLES: {linebreak.join(df[0]['inferrables'].to_list()[0])}\n
CAUSAL RELATIONSHIPS: {linebreak.join(df[0]['causal_relationships'].to_list())}
""")



In [ ]:
import numpy as np
"\n".join(np.array(df["speculatives"].list.sample(1).sample(5).to_list()).flatten())

In [ ]:
df = pl.read_parquet(data_dir / "conversation_summaries/cm0i27jdj0000aqpa73ghpcxf.snappy")
d = show(df.to_pandas())
d.open_browser()

In [ ]:
df = pl.read_parquet(data_dir / "conversations_clusters_summaries/cm0i27jdj0000aqpa73ghpcxf.snappy")
d = show(df.to_pandas())
d.open_browser()

In [ ]:
r = (df.filter(pl.col("is_emotional")) 
    .with_columns(
        title_emotions=pl.concat_str(
                [
                    pl.col("start_date"),
                    pl.lit(" - "),
                    pl.col("title"),
                    pl.lit(":\n- "),
                    pl.col("strong_emotional_implications").list.join("\n- "),
                    pl.lit("\n\n"),
                ]
            )
        )
        .group_by("fine_cluster_label", "fine_cluster_summary", "coarse_cluster_label")
        .agg(
            title_emotions=pl.col("title_emotions").str.concat("\n"),
            average_cluster_date=pl.col("start_date").str.to_date().mean(),
        )
        .with_columns(
            cluster_emotions=pl.concat_str(
                [
                    pl.lit("### "),
                    pl.col("fine_cluster_label"),
                    pl.lit(" - "),
                    pl.col("fine_cluster_summary"),
                    pl.lit(":\n\n"),
                    pl.col("title_emotions"),
                ]
            )
        )
        .group_by("coarse_cluster_label")
        .agg(
            cluster_emotions=pl.col("cluster_emotions").str.concat("\n"),
            average_cluster_date=pl.col("average_cluster_date").dt.date().mean(),
        )
        .sort("average_cluster_date"))
print(r.filter(pl.col("coarse_cluster_label") == 21).get_column("cluster_emotions")[0])



In [ ]:
from pathlib import Path
import pandas as pd
from dtale import show
import dtale.global_state as global_state

global_state.set_app_settings(dict(max_column_width=300))

data_dir = Path().absolute() / ".." / "data"
df = pd.read_json(data_dir / "conversation_clusters/cm0i27jdj0000aqpa73ghpcxf.snappy")
d = show(df)
d.open_browser()

In [ ]:
import numpy as np
i,c = np.unique(df.get_column("cluster_category").fill_null(""), return_counts=True)
for pair in zip(i, c):
    print(pair[1], pair[0])

In [ ]:
COL = "cluster_label"

df = df.sort(COL, "date", descending=[False, False])

res = df.with_columns(
        pl.concat_str(
            [
                pl.when(pl.col("interests_quirkiness").eq(True))
                .then(pl.lit("NB:"))
                .otherwise(pl.lit("")),
                pl.col("date"),
                pl.lit(":"),
                pl.col("interests"),
            ],
        ).alias("date_interests")
).group_by(
    COL
).agg(
    # Aggregate the interests with their dates into a list of strings
    pl.col("date_interests").str.concat("\n").alias("cluster_items"),
).filter(
    pl.col(COL) == 1238
).get_column("cluster_items")[0]

print(res)
#1689

In [ ]:
import umap
import plotly.express as px

In [ ]:
df = df.filter(pl.col("fine_centroid").is_not_null()).group_by("fine_dissimilarity_rank").agg(pl.col("fine_centroid").first())

# Reduce dimensionality of fine_centroid embeddings
reducer = umap.UMAP(n_components=2, random_state=42)
embeddings_2d = reducer.fit_transform(df['fine_centroid'].to_list())

# Create a new dataframe with reduced embeddings and labels
plot_df = pl.DataFrame({
    'x': embeddings_2d[:, 0],
    'y': embeddings_2d[:, 1],
    'rank': df['fine_dissimilarity_rank']
})

# Create an interactive scatter plot
fig = px.scatter(
    plot_df.to_pandas(),
    x='x',
    y='y',
    color='rank',
    hover_data=['rank'],
    title='Fine Centroid Embeddings Visualization'
)

# Show the plot
fig.show()

In [ ]:
from sklearn.metrics import pairwise_distances
import numpy as np

# Create some sample data
X = np.array([[-9999, -9999], [1, 1], [-1, -1], [9999, 9999]])

# Calculate pairwise distances using Euclidean distance (default metric)
distances = pairwise_distances(X, metric='cosine')

print("Pairwise distances:")
print(distances)

# Calculate pairwise distances using Manhattan distance
manhattan_distances = pairwise_distances(X, metric='manhattan')

print("\nPairwise Manhattan distances:")
print(manhattan_distances)